### Few shot prompt engernering

In [ ]:
!pip install -q -U trl==0.12 transformers accelerate
!pip install -q -U datasets bitsandbytes

In [ ]:
import torch
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, AutoTokenizer

model_name = 'cjvt/GaMS-9B-Instruct'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, # loading in 4 bit
    bnb_4bit_quant_type="nf4", # quantization type
    bnb_4bit_use_double_quant=True, # nested quantization
    bnb_4bit_compute_dtype=torch.bfloat16,
)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    trust_remote_code=True
)
model.config.use_cache = False

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

# Few Shot prompt engeneering

In [ ]:
generator = pipeline(
    model=model, tokenizer=tokenizer,
    task='text-generation',
    temperature=0.001,
    max_new_tokens=500,
    repetition_penalty=1.1
)

In [ ]:
import pandas as pd

REPORTS = pd.read_csv("traffic_reports_linked.csv", encoding='utf-8')
#REPORTS.head()

In [ ]:
RTFS = pd.read_csv("rtfs_reduced.csv", encoding='utf-8')
#RTFS.head()

In [ ]:
from sklearn.model_selection import train_test_split

RTFS_train, temp = train_test_split(RTFS, test_size=0.3, random_state=42)
RTFS_valid, RTFS_test = train_test_split(temp, test_size=0.5, random_state=42)
print(f"RTFS size: {len(RTFS)}")
print(f"RTFS_train size: {len(RTFS_train)}")
print(f"RTFS_valid size: {len(RTFS_valid)}")
print(f"RTFS_test size: {len(RTFS_test)}")

In [ ]:
def prepare_input(RTF_file_name):

    # Traffic data for the given RTF file name
    reports = REPORTS[REPORTS['RTF_file_name'] == RTF_file_name]

    # Input data used for generating desired LLM output
    input = {col: set(reports[col].dropna()) for col in [
        'Datum',
        'A1', 
        'B1', 
        'ContentPomembnoSLO', 
        'ContentNesreceSLO', 
        'ContentZastojiSLO', 
        'ContentVremeSLO', 
        'ContentOvireSLO', 
        'ContentDeloNaCestiSLO', 
        'ContentOpozorilaSLO',
        'ContentMednarodneInformacijeSLO', 
        'ContentSplosnoSLO']}
    
    lines = ["### Vhodni podatki:"]
    
    if input['ContentPomembnoSLO']:
        lines.append(f"- Zelo pomembne informacije o prometu: {', '.join(map(str, input['ContentPomembnoSLO']))}")
    if input['A1']:
        lines.append(f"- Pomembne informacije o prometu: {', '.join(map(str, input['A1']))}")
    if input['B1']:
        lines.append(f"- Manj pomembne informacije o prometu: {', '.join(map(str, input['B1']))}")
    if input['ContentNesreceSLO']:
        lines.append(f"- Informacije o nesrečah: {', '.join(map(str, input['ContentNesreceSLO']))}")
    if input['ContentZastojiSLO']:
        lines.append(f"- Informacije o zastojih: {', '.join(map(str, input['ContentZastojiSLO']))}")
    if input['ContentVremeSLO']:
        lines.append(f"- Informacije o vremenu: {', '.join(map(str, input['ContentVremeSLO']))}")
    if input['ContentOvireSLO']:
        lines.append(f"- Informacije o ovirah: {', '.join(map(str, input['ContentOvireSLO']))}")
    if input['ContentDeloNaCestiSLO']:
        lines.append(f"- Informacije o delu na cesti: {', '.join(map(str, input['ContentDeloNaCestiSLO']))}")
    if input['ContentOpozorilaSLO']:
        lines.append(f"- Informacije o opozorilih: {', '.join(map(str, input['ContentOpozorilaSLO']))}")
    if input['ContentMednarodneInformacijeSLO']:
        lines.append(f"- Informacije o mednarodnih informacijah: {', '.join(map(str, input['ContentMednarodneInformacijeSLO']))}")
    if input['ContentSplosnoSLO']:
        lines.append(f"- Splošne informacije: {', '.join(map(str, input['ContentSplosnoSLO']))}")
    
    if lines == ["### Vhodni podatki:"]:
        return None

    return lines

def generate_shot(RTFS_, RTF_file_name):
    
    # Preparing example shot text
    shot = prepare_input(RTF_file_name)

    if shot == None:
        return None

    # Desired LLM output
    output = RTFS_[RTFS_['file_name'] == RTF_file_name]["content"].values[0]

    shot.append("")
    shot.append(f"### Poročilo: {output}\n")

    return '\n'.join(shot)

# Example usage
print(generate_shot(RTFS_train, 2))


In [ ]:
# Randomly select a test RTF file name and prepare the input data
random_test_RTF = RTFS_test.sample(1).iloc[0]['file_name']

# Ipnut data lines for the test sample
test_data = '\n'.join(prepare_input(random_test_RTF))

# Generate the few-shot prompt with some training examples
few_shot_prompt = f"""Generiraj prometno poročilo na podlagi spodnjih vhodnih podatkov:
{test_data}

Spodaj je podanih par primerov vhodnih poročil in željenih izhodnih poročil iz teh podatkov:

{generate_shot(RTFS_train, 1)}
{generate_shot(RTFS_train, 2)}
{generate_shot(RTFS_train, 3)}
"""
# Example of few-shot prompt
#print(few_shot_prompt)
with open("few_shot_prompt.txt", "w", encoding="utf-8") as f:
    f.write(few_shot_prompt)

In [ ]:
res = generator(few_shot_prompt)
print(res[0]["generated_text"])

Generate dataset for training, testing and validation (prepare prompts).

In [ ]:
from datasets import Dataset

RTFS_train["prompt"] = None
RTFS_test["prompt"] = None
RTFS_valid["prompt"] = None
for index, row in RTFS_train.iterrows():
    shot = generate_shot(RTFS_train, row['file_name'])
    if shot:
        RTFS_train.at[index, 'prompt'] =  shot

for index, row in RTFS_test.iterrows():
    shot = generate_shot(RTFS_test, row['file_name'])
    if shot:
        RTFS_test.at[index, 'prompt'] =  shot

for index, row in RTFS_valid.iterrows():
    shot = generate_shot(RTFS_valid, row['file_name'])
    if shot:
        RTFS_valid.at[index, 'prompt'] =  shot

train_dataset = RTFS_train[RTFS_train["prompt"].notnull()]["prompt"]
test_dataset = RTFS_test[RTFS_test["prompt"].notnull()]["prompt"]
valid_dataset = RTFS_valid[RTFS_valid["prompt"].notnull()]["prompt"]

train_dataset = Dataset.from_pandas(train_dataset)
test_dataset = Dataset.from_pandas(test_dataset)
valid_dataset = Dataset.from_pandas(valid_dataset)



# Low_Rank adaptation (LoRA)

In [ ]:
from peft import LoraConfig, get_peft_model

lora_alpha = 32
lora_dropout = 0.1
lora_r = 16

peft_config = LoraConfig(
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    r=lora_r,
    bias="none",
    task_type="CAUSAL_LM"
)

In [ ]:
def print_trainable_parameters(model):
    """
    Prints the number of trainable parameters in the model.
    """
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(
        f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param}"
    )

In [ ]:
model

In [ ]:
lora_model = get_peft_model(model, peft_config)

In [ ]:
lora_model

In [ ]:
print_trainable_parameters(lora_model)

## Training

Training was done using train.py. See that for details.